# PreProcess Notebook

This notebook is the Jupyter version of PreProcess.py. It converts PE files to JSON, builds an external function vocabulary, and converts JSON data to PyG objects.

In [ ]:
import json
import torch
from torch_geometric.data import Data
from tqdm import tqdm
import os
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor

from r2_acfg_extractor import R2ACFGExtractor
from BuildExternalVocab import ExternalVocabBuilder
from Vocabulary import Vocab

In [ ]:
# Convert PE files to JSON format and build the vocabulary of external function names
def iter_dataset_files(dataset_path: str):
    for root, _, files in os.walk(dataset_path):
        for file in files:
            file_path = os.path.join(root, file)
            if os.path.isfile(file_path):
                yield file_path


def _process_pe_file(job):
    input_file, dataset_path, output_dir = job
    output_file = input_file.replace(dataset_path, output_dir) + ".json"

    # Preserve directory structure for nested dataset paths.
    os.makedirs(os.path.dirname(output_file), exist_ok=True)

    if os.path.exists(output_file):
        return

    extractor = R2ACFGExtractor(binary_path=input_file, output_path=output_file)
    extractor.run()


def pe_to_json(dataset_path: str, output_dir: str):
    os.makedirs(output_dir, exist_ok=True)
    dataset_files = list(iter_dataset_files(dataset_path))
    jobs = [(input_file, dataset_path, output_dir) for input_file in dataset_files]

    # Use ProcessPoolExecutor to process files in parallel.
    with ProcessPoolExecutor(max_workers=12) as executor:
        list(
            tqdm(
                executor.map(_process_pe_file, jobs),
                total=len(dataset_files),
                desc="Processing files (PE to JSON)"
            )
        )


def fix_external_function_indices(dataset_path: str, vocab: Vocab):
    for file in tqdm(iter_dataset_files(dataset_path), desc="Fixing external function indices"):
        with open(file, "r", encoding="utf-8") as f:
            data = json.load(f)
            function_names = data.get("function_names", [])
            function_edges = data.get("function_edges", [])

            if not function_names or not function_edges:
                continue

            index_to_name = {index: name for index, name in enumerate(function_names)}

            old_to_new = {}
            for index, name in index_to_name.items():
                if name in vocab.token_2_index:
                    old_to_new[index] = vocab.token_2_index[name]

            new_function_edges = []
            for indexes in function_edges:
                new_indexes = [old_to_new.get(index, index) for index in indexes]
                new_function_edges.append(new_indexes)

            data["function_edges"] = new_function_edges

        with open(file, "w", encoding="utf-8") as f:
            json.dump(data, f, ensure_ascii=True)


def parse_json_list_2_pyg_object(jsonl_file: str, label: int, vocab: Vocab, output_file: str):
    os.makedirs(os.path.dirname(output_file), exist_ok=True)
    with open(jsonl_file, "r", encoding="utf-8") as file:
        for item in tqdm(file):
            item = json.loads(item)
            item_hash = item['hash']

            acfg_list = []
            for one_acfg in item['acfg_list']:
                block_features = one_acfg['block_features']
                block_edges = one_acfg['block_edges']
                one_acfg_data = Data(
                    x=torch.tensor(block_features, dtype=torch.float),
                    edge_index=torch.tensor(block_edges, dtype=torch.long)
                )
                acfg_list.append(one_acfg_data)

            item_function_names = item['function_names']
            item_function_edges = item['function_edges']

            local_function_name_list = item_function_names[:len(acfg_list)]
            assert len(acfg_list) == len(local_function_name_list), (
                "The length of ACFG_List should be equal to the length of Local_Function_List"
            )
            external_function_name_list = item_function_names[len(acfg_list):]

            external_function_index_list = [vocab[f_name] for f_name in external_function_name_list]
            torch.save(
                Data(
                    hash=item_hash,
                    local_acfgs=acfg_list,
                    external_list=external_function_index_list,
                    function_edges=item_function_edges,
                    targets=label,
                ),
                output_file,
            )

In [ ]:
# Configuration (equivalent to argparse defaults in PreProcess.py)
dataset_path = "Datasets/pe-machine-learning-dataset/divided_dataset"
output_dir_json = "Datasets/pe-machine-learning-dataset/graph_data"
output_dir_pyg = "Datasets/pe-machine-learning-dataset/pyg_data"
vocab_file = "./train_external_function_name_vocab.jsonl"
max_vocab_size = 10000
label = 1

os.makedirs(output_dir_json, exist_ok=True)
os.makedirs(output_dir_pyg, exist_ok=True)

In [ ]:
# Convert PE files to JSON format
pe_to_json(dataset_path=dataset_path, output_dir=output_dir_json)
print("PE to JSON conversion completed.")

In [ ]:
# Build vocabulary once from all generated JSON files
external_vocab_builder = ExternalVocabBuilder(input_path=output_dir_json, output_file=vocab_file)
external_vocab_builder.run()

vocabulary = Vocab(freq_file=vocab_file, max_vocab_size=max_vocab_size)
print("Vocabulary built with size:", len(vocabulary))

In [ ]:
# Normalize function_edges using vocabulary indices before PyG conversion
fix_external_function_indices(dataset_path=output_dir_json, vocab=vocabulary)
print("External function indices fixed.")

In [ ]:
# Cycle for all the JSON files in the output directory and convert them to PyG objects
for root, _, files in os.walk(output_dir_json):
    with ThreadPoolExecutor(max_workers=12) as executor:
        json_files = [f for f in files if f.endswith(".json")]

        def _convert_one(json_file: str):
            json_file_path = os.path.join(root, json_file)
            output_file = os.path.join(
                root.replace(output_dir_json, output_dir_pyg),
                json_file.replace(".json", ".pt"),
            )

            if os.path.exists(output_file):
                return

            parse_json_list_2_pyg_object(
                jsonl_file=json_file_path,
                label=label,
                vocab=vocabulary,
                output_file=output_file,
            )

        list(
            tqdm(
                executor.map(_convert_one, json_files),
                total=len(json_files),
                desc="Converting JSON to PyG objects",
                leave=False,
            )
        )

print("Preprocessing complete.")